# 174 — Head QK Analysis

Analyze attention head query-key circuits: alignment profiles,
positional vs content attention, selectivity metrics, QK subspace
structure, and attention pattern classification.

## Why This Matters

QK circuit head analyzes the query-key interaction that determines attention patterns. Understanding what features drive attention — positional vs. content-based, local vs. global — reveals how the model decides which information to route where.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.head_qk_analysis import (
    qk_alignment_profile,
    positional_vs_content_attention,
    attention_selectivity,
    key_query_subspace,
    attention_pattern_type,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.arange(15)

In [ ]:
result = qk_alignment_profile(model, tokens, layer=0)
for h in result['per_head']:
    print(f"Head {h['head']}: q_norm={h['q_norm']:.3f}  k_norm={h['k_norm']:.3f}  "
          f"dot_mean={h['dot_product_mean']:.4f}  dot_std={h['dot_product_std']:.4f}")

In [ ]:
result = positional_vs_content_attention(model, tokens, layer=0)
for h in result['per_head']:
    kind = 'POSITIONAL' if h['is_positional'] else 'CONTENT'
    print(f"Head {h['head']}: {kind:10s}  pos_var={h['positional_variance']:.4f}  "
          f"cont_var={h['content_variance']:.4f}  score={h['positional_score']:.3f}")

In [ ]:
result = attention_selectivity(model, tokens, layer=0)
for h in result['per_head']:
    print(f"Head {h['head']}: entropy={h['mean_entropy']:.3f}  "
          f"norm_entropy={h['normalized_entropy']:.3f}  "
          f"max_wt={h['mean_max_weight']:.3f}  gini={h['gini_coefficient']:.3f}")

In [ ]:
result = key_query_subspace(model, layer=0)
for h in result['per_head']:
    print(f"Head {h['head']}: eff_rank={h['effective_rank']:.2f}  "
          f"cond={h['condition_number']:.2f}  top_sv={h['top_singular_value']:.4f}")

In [ ]:
result = attention_pattern_type(model, tokens, layer=0)
for h in result['per_head']:
    print(f"Head {h['head']}: type={h['pattern_type']:15s}  "
          f"diag={h['diagonal_score']:.3f}  prev={h['previous_token_score']:.3f}  "
          f"sparse={h['sparse_score']:.3f}")